# MedSAM2 fine-tuning — ground-truth evaluation on held-out cases

> **5-fold mode.** This notebook now evaluates the 5-fold leave-subjects-out
> cross-validation built by `build_npz_dataset.py --kfold 5`
> (`MedSAM2_finetune_data/kfold/fold_k/`). Fold *k*'s checkpoint
> (`exp_log/kfold_k`) is scored only on fold *k*'s holdout subjects, so every
> labeled case is scored exactly once by a model that never trained on its
> subject. Results are written to `3D_eval_holdout_5fold.xlsx`. Because held-out
> subjects have no cases in training, the `seen_subject` tier is empty here.


Scores a fine-tuned checkpoint against the manually annotated single-frame 3D
volumes under `MedSAM2_finetune_data/processed/`, implementing Step 6 of
`medsam2_function/README.md`.

Metrics come from **MONAI** (`monai.metrics`) — Dice, IoU, average symmetric
surface distance, 95th-percentile Hausdorff, and normalized surface Dice — so
they match the definitions used elsewhere in the medical-imaging literature.
Each case's own `spacing_mm_xyz` is handed to MONAI as `spacing=`, making every
distance a physical millimetre rather than a voxel count; these volumes are
anisotropic (~0.31 × 0.57 × 0.42 mm), so that distinction is not cosmetic.

### What gets reported, and what it means

Cases are graded into three tiers, derived from **which `.npz` files exist in
`npz_train/`** — that folder is the only trustworthy record of what the trainer
actually consumed. (Every `image.json` in the tree says `recommended_split:
"train"`, including cases never trained on, so that field is stale and is
ignored here.)

| tier | meaning | supports a generalization claim? |
|---|---|---|
| `holdout` | the **subject** never appears in `npz_train` | **yes — this is the headline number** |
| `seen_subject` | different bolus of a subject that *was* trained on | weak: same patient, same lesion |
| `train` | this exact case was trained on | no — fit check only |

### Two caveats to carry into any write-up

**1. Every score is prompt-conditional.** `compute_3d_mask()` is a *prompted*
segmenter — it needs a 3D box. In the 4D pipeline that box comes from the
motion-tracked reference VOI; these extracted frames have no tracked bbox, so
the box is derived from the GT mask itself. That measures *segmentation given a
correct box*, not detection, and it is an optimistic setting. The jitter sweep
near the bottom probes how fast that advantage decays.

**2. n = 3 held-out cases, from 2 subjects.** Report per-case numbers, not just
a mean. A 3-case mean has no meaningful confidence interval, and two of the
three cases share a subject (and therefore a lesion).

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Deliberately does NOT chdir. Paths derive from one absolute anchor, so
# re-running this cell any number of times is a no-op rather than walking the
# working directory a level further up each time.
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

QUANTUS_ROOT = Path("/home/ahmed-el-kaffas/Documents/Github/QuantUS")

MEDSAM2_ROOT = QUANTUS_ROOT / "MedSAM2"
CLI_DEMOS    = QUANTUS_ROOT / "engines" / "ceus" / "CLI-Demos"
FUNCTIONS    = CLI_DEMOS / "medsam2_function"

for path in (MEDSAM2_ROOT, CLI_DEMOS, FUNCTIONS):
    assert path.is_dir(), f"not found: {path}"
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import medsam2_gt_eval as gte
from medsam2_viewer_base import enhance_bmode_noise

print(f"MedSAM2 root : {MEDSAM2_ROOT}")
print(f"functions    : {FUNCTIONS}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
FINETUNE_ROOT  = Path("/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_finetune_data")
PROCESSED_ROOT = FINETUNE_ROOT / "processed"
KFOLD_ROOT     = FINETUNE_ROOT / "kfold"          # fold_k/npz_train defines each fold's split
N_FOLDS        = 5
OUT_XLSX       = FINETUNE_ROOT / "3D_eval_holdout_5fold.xlsx"

MODEL_CFG = "configs/sam2.1_hiera_t512.yaml"
DEVICE    = "cuda"

# One epoch for every fold, chosen up front. Picking the best epoch per fold on
# the same holdout would make the pooled score optimistic.
FINETUNED_CKPT_NAME = "checkpoint.pt"             # last epoch; or e.g. "checkpoint_30.pt"
FINETUNED_LABEL     = "finetuned (5-fold)"
# The baseline is the model the fine-tuning started from (see the training YAML's
# checkpoint_path), so the comparison isolates what fine-tuning added.
BASELINE_CKPT  = MEDSAM2_ROOT / "checkpoints" / "MedSAM2_MRI_LiverLesion.pt"
BASELINE_LABEL = "base (MedSAM2_MRI_LiverLesion)"
ORDER = [BASELINE_LABEL, FINETUNED_LABEL]

# Reconstruction settings — keep these matched to Medsam2AdaptiveBboxMasker's
# defaults so the score reflects what the 4D pipeline actually produces.
PADDING_VOX     = 4
SMOOTH_SIGMA_MM = 1.0

# NSD tolerance, in mm. 1.0 mm lands in a useful range on this data — measured
# NSD sits near 0.5 while Dice is near 0.85, so it separates cases that Dice
# rates similarly. Report the tolerance alongside the number: NSD is meaningless
# without it, and the value should ideally be justified by inter-observer
# variability on these annotations rather than picked for convenience.
TOLERANCE_MM = 1.0

## Cases and split

In [ ]:
fold_cases = {}
for k in range(1, N_FOLDS + 1):
    fold_cases[k] = gte.discover_cases(str(PROCESSED_ROOT), str(KFOLD_ROOT / f"fold_{k}" / "npz_train"))

case_df = pd.DataFrame([
    {"fold": k, **{key: c[key] for key in ("case", "subject_id", "sequence_id", "tier")}}
    for k, cases_k in fold_cases.items() for c in cases_k
])
held = case_df[case_df.tier == "holdout"]

# Sanity checks: this is a leave-subjects-out CV, so every labeled case must be
# held out in exactly one fold, and nothing may be tiered "seen_subject".
assert (case_df.tier == "seen_subject").sum() == 0, "a held-out subject also has cases in training"
assert held.case.is_unique, "a case is held out in more than one fold"
assert held.case.nunique() == case_df.case.nunique(), "some case is never held out"

print(f"{held.case.nunique()} cases over {held.subject_id.nunique()} subjects, each held out once\n")
display(held.groupby("fold").agg(subjects=("subject_id", lambda s: ", ".join(sorted(set(s)))),
                                  n_cases=("case", "count")))

## Checkpoints to compare

For each fold: the **baseline** (the model fine-tuning started from) and that
fold's **fine-tuned** checkpoint, both scored on the same held-out cases. The
baseline is what makes a Dice of 0.8 interpretable. Only one fine-tuned epoch is
used (`FINETUNED_CKPT_NAME`), fixed before looking at any scores.

In [ ]:
import os

fold_ckpts = {}
for k in range(1, N_FOLDS + 1):
    fold_ckpts[k] = {
        BASELINE_LABEL:  Path(BASELINE_CKPT),
        FINETUNED_LABEL: MEDSAM2_ROOT / "exp_log" / f"kfold_{k}" / "checkpoints" / FINETUNED_CKPT_NAME,
    }

for k, ckpts in fold_ckpts.items():
    npz_dir = KFOLD_ROOT / f"fold_{k}" / "npz_train"
    newest_npz = max(os.path.getmtime(p) for p in npz_dir.glob("*.npz"))
    for label, path in ckpts.items():
        ok = Path(path).is_file()
        # a fine-tuned checkpoint older than its training .npz files was trained on stale data
        stale = ok and label == FINETUNED_LABEL and os.path.getmtime(path) < newest_npz
        print(f"fold {k}  {'OK     ' if ok else 'MISSING'}{' STALE (older than npz_train)' if stale else ''}  {label:<30} {path}")

## Evaluate — held-out tier

In [ ]:
rows = []
for k in range(1, N_FOLDS + 1):
    for label, path in fold_ckpts[k].items():
        if not Path(path).is_file():
            print(f"skipping missing checkpoint: fold {k} {label}")
            continue
        batch = gte.evaluate_checkpoint(
            fold_cases[k], MODEL_CFG, str(path), device=DEVICE, tiers=("holdout",),
            padding_vox=PADDING_VOX, smooth_sigma_mm=SMOOTH_SIGMA_MM,
            tolerance_mm=TOLERANCE_MM, label=label,
        )
        for row in batch:
            row["fold"] = k
        rows += batch

results = pd.DataFrame(rows)
print(f"\n{len(results)} rows ({results.case.nunique()} cases)")

In [ ]:
# ── Per-case detail ──────────────────────────────────────────────────────────
# Individual values carry the information; a mean alone would hide a single
# catastrophic case behind good ones.
nsd_col = f"nsd@{TOLERANCE_MM}mm"
metric_cols = ["dice", "iou", nsd_col, "assd_mm", "hd95_mm", "vol_ratio"]

display(
    results.pivot_table(index="case", columns="checkpoint", values="dice")
           .style.format("{:.3f}").set_caption("Dice per case")
)
display(results[["fold", "checkpoint", "case", *metric_cols, "empty_pred"]].round(3))

In [ ]:
# ── Aggregate ────────────────────────────────────────────────────────────────
# Headline: mean/std over ALL held-out cases (each scored once), not over fold means —
# folds differ in size, and averaging fold means would over-weight small folds.
agg = results.groupby("checkpoint")[metric_cols].agg(["mean", "std"]).round(3)
agg = agg.reindex([c for c in ORDER if c in agg.index])
display(agg)

per_fold = results.groupby(["fold", "checkpoint"])[metric_cols].mean().round(3)
display(per_fold)

# Per subject: how much each patient contributes, and whether one drives the mean.
per_subject = results.groupby(["subject_id", "checkpoint"])[metric_cols].mean().round(3)

# Paired difference (fine-tuned minus baseline) per case; positive dice_delta = fine-tuning helped.
wide = results.pivot_table(index=["fold", "subject_id", "case"], columns="checkpoint", values="dice")
paired = wide.assign(dice_delta=wide[FINETUNED_LABEL] - wide[BASELINE_LABEL]).round(3).reset_index()
print(f"fine-tuned better on {(paired.dice_delta > 0).sum()} of {len(paired)} cases; "
      f"mean dice delta {paired.dice_delta.mean():+.3f}")

if results["empty_pred"].any():
    failed = results[results.empty_pred][["fold", "checkpoint", "case"]]
    print("\nEmpty predictions (surface metrics are NaN for these):")
    print(failed.to_string(index=False))

In [ ]:
# ── Export to Excel ──────────────────────────────────────────────────────────
try:
    import openpyxl  # noqa: F401  (pandas' .xlsx writer)
except ImportError as e:
    raise ImportError("pip install openpyxl in this kernel's environment to write the .xlsx") from e

settings = pd.DataFrame({"setting": [
    "finetuned checkpoint", "baseline checkpoint", "n_folds", "padding_vox", "smooth_sigma_mm",
    "nsd tolerance (mm)", "box prompt"], "value": [
    FINETUNED_CKPT_NAME, str(BASELINE_CKPT), N_FOLDS, PADDING_VOX, SMOOTH_SIGMA_MM,
    TOLERANCE_MM, "derived from the GT mask (optimistic, see header)"]})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    results.sort_values(["fold", "checkpoint", "case"]).to_excel(xw, "per_case", index=False)
    agg.to_excel(xw, "summary_all_cases")
    per_fold.to_excel(xw, "per_fold")
    per_subject.to_excel(xw, "per_subject")
    paired.to_excel(xw, "paired_dice", index=False)
    held.sort_values(["fold", "case"]).to_excel(xw, "fold_assignment", index=False)
    settings.to_excel(xw, "settings", index=False)
print(f"wrote {OUT_XLSX}")

## Epoch sweep — every checkpoint of every fold

There is no validation loop in the training config, so no automatic "best
epoch". This scores `checkpoint_10 … checkpoint_50` of each fold on that fold's
held-out cases, so the epoch can be chosen from the pooled cross-validation
curve. Choose **one epoch for all folds** (and for any final all-data model);
picking a different best epoch per fold on the same holdout inflates the score.
Once an epoch is chosen from these scores, the pooled number is slightly
optimistic — say so when reporting it. Slow: 5 epochs × 41 cases.

In [ ]:
EPOCHS = [10, 20, 30, 40, 50]

sweep_rows = []
for k in range(1, N_FOLDS + 1):
    for epoch in EPOCHS:
        path = MEDSAM2_ROOT / "exp_log" / f"kfold_{k}" / "checkpoints" / f"checkpoint_{epoch}.pt"
        if not path.is_file():
            print(f"skipping missing checkpoint: fold {k} epoch {epoch}")
            continue
        batch = gte.evaluate_checkpoint(
            fold_cases[k], MODEL_CFG, str(path), device=DEVICE, tiers=("holdout",),
            padding_vox=PADDING_VOX, smooth_sigma_mm=SMOOTH_SIGMA_MM,
            tolerance_mm=TOLERANCE_MM, label=f"fold{k} ep{epoch}", verbose=False,
        )
        for row in batch:
            row["fold"], row["epoch"] = k, epoch
        sweep_rows += batch
        print(f"fold {k} epoch {epoch:>2}: mean dice {np.mean([r['dice'] for r in batch]):.3f} ({len(batch)} cases)")

sweep_df = pd.DataFrame(sweep_rows)

In [ ]:
# ── Epoch summary, plot, and export ──────────────────────────────────────────
# Pooled over ALL held-out cases (each scored once per epoch), not over fold means.
epoch_summary = sweep_df.groupby("epoch")[metric_cols].agg(["mean", "std"]).round(3)
display(epoch_summary)

epoch_per_fold = sweep_df.pivot_table(index="fold", columns="epoch", values="dice", aggfunc="mean").round(3)
display(epoch_per_fold.style.format("{:.3f}").set_caption("Mean Dice per fold and epoch"))

best_epoch = int(sweep_df.groupby("epoch").dice.mean().idxmax())
base_dice = results[results.checkpoint == BASELINE_LABEL].dice.mean()
print(f"best pooled epoch by mean Dice: {best_epoch}   (baseline mean Dice: {base_dice:.3f})")
print(f"-> set FINETUNED_CKPT_NAME = 'checkpoint_{best_epoch}.pt' in the configuration cell to use it everywhere")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for k, grp in sweep_df.groupby("fold"):
    axes[0].plot(*zip(*sorted(grp.groupby("epoch").dice.mean().items())), marker="o", alpha=0.6, label=f"fold {k}")
axes[0].plot(*zip(*sorted(sweep_df.groupby("epoch").dice.mean().items())), color="k", lw=2.5, marker="s", label="pooled")
axes[0].axhline(base_dice, color="gray", ls="--", label="baseline")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("mean Dice"); axes[0].set_title("Dice vs epoch"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7)
for metric in ("assd_mm", "hd95_mm"):
    axes[1].plot(*zip(*sorted(sweep_df.groupby("epoch")[metric].mean().items())), marker="o", label=metric)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("mm"); axes[1].set_title("Surface distance vs epoch (pooled)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Append to the same workbook the main evaluation wrote (create it if that cell hasn't run).
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl", mode="a" if OUT_XLSX.exists() else "w",
                    **({"if_sheet_exists": "replace"} if OUT_XLSX.exists() else {})) as xw:
    sweep_df.sort_values(["fold", "epoch", "case"]).to_excel(xw, "epoch_per_case", index=False)
    epoch_summary.to_excel(xw, "epoch_summary")
    epoch_per_fold.to_excel(xw, "epoch_per_fold")
print(f"updated {OUT_XLSX}")

In [ ]:
# ── Metric vs. checkpoint ────────────────────────────────────────────────────
plot_metrics = ["dice", nsd_col, "assd_mm", "hd95_mm"]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4 * len(plot_metrics), 3.6))

order = [c for c in ORDER if c in results.checkpoint.unique()]
x = np.arange(len(order))

for ax, metric in zip(axes, plot_metrics):
    # one line per case, so a checkpoint that helps one case and wrecks another
    # is visible rather than averaged away
    for case, grp in results.groupby("case"):
        grp = grp.set_index("checkpoint").reindex(order)
        ax.plot(x, grp[metric], marker="o", alpha=0.65, label=case)
    means = [results[results.checkpoint == c][metric].mean() for c in order]
    ax.plot(x, means, color="k", linewidth=2.5, marker="s", label="mean")

    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    if metric.endswith("_mm"):
        ax.set_ylabel("mm")

axes[0].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

## Prompt sensitivity

The box above was derived from the GT mask, so these scores are the optimistic
end of the range. Jittering each box face by a few voxels approximates a box
that came from imperfect tracking instead — the slope of this curve tells you
how much of the performance is the model and how much was handed to it by the
prompt. A model whose Dice collapses at 8 voxels of jitter will not survive
contact with the tracked bbox in the 4D pipeline. Each case is scored by the
checkpoint of the fold that held it out.

In [ ]:
JITTER_SWEEP = [0, 2, 4, 8]

jitter_rows = []
for jitter in JITTER_SWEEP:
    batch = []
    for k in range(1, N_FOLDS + 1):
        batch += gte.evaluate_checkpoint(
            fold_cases[k], MODEL_CFG, str(fold_ckpts[k][FINETUNED_LABEL]), device=DEVICE,
            tiers=("holdout",), padding_vox=PADDING_VOX, jitter_vox=jitter,
            smooth_sigma_mm=SMOOTH_SIGMA_MM, tolerance_mm=TOLERANCE_MM,
            label=f"jitter={jitter}", seed=0, verbose=False,
        )
    for row in batch:
        row["jitter_vox"] = jitter
    jitter_rows += batch
    print(f"jitter {jitter:>2} vox — mean dice over {len(batch)} cases: {np.mean([r['dice'] for r in batch]):.3f}")

jitter_df = pd.DataFrame(jitter_rows)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
for case, grp in jitter_df.groupby("case"):
    ax.plot(grp.sort_values("jitter_vox").jitter_vox, grp.sort_values("jitter_vox").dice,
            marker="o", alpha=0.25, color="gray", lw=1)
ax.plot(jitter_df.groupby("jitter_vox").dice.mean(), color="k", lw=2.5, marker="s", label="mean")
ax.set_xlabel("box jitter (voxels, per face)")
ax.set_ylabel("Dice")
ax.set_title(f"Prompt sensitivity — {FINETUNED_LABEL}")
ax.grid(alpha=0.3)
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

jitter_df.groupby("jitter_vox")[metric_cols].mean().round(3).to_csv(FINETUNE_ROOT / "3D_eval_jitter_5fold.csv")

## Qualitative check

Metrics hide failure *modes*. This shows the GT contour against the prediction
on the mask's central axial slice for held-out cases (up to `N_SHOW` per fold),
which is where an over-segmentation into adjacent liver, or a mask that
collapsed to the box, is immediately obvious.

In [ ]:
# One row per fold: up to N_SHOW held-out cases of that fold, scored by that fold's checkpoint.
N_SHOW = 4
fig, axes = plt.subplots(N_FOLDS, N_SHOW, figsize=(4 * N_SHOW, 4 * N_FOLDS), squeeze=False)

for k in range(1, N_FOLDS + 1):
    predictor = gte.build_predictor(MODEL_CFG, str(fold_ckpts[k][FINETUNED_LABEL]), device=DEVICE)
    holdout = [c for c in fold_cases[k] if c["tier"] == "holdout"][:N_SHOW]
    for ax in axes[k - 1]:
        ax.axis("off")
    for ax, case in zip(axes[k - 1], holdout):
        volume, gt, spacing, _ = gte.load_case(case)
        pred, bbox = gte.predict_case(
            predictor, volume, gt, padding_vox=PADDING_VOX,
            spacing=spacing, smooth_sigma_mm=SMOOTH_SIGMA_MM,
        )
        z = int(np.argmax(gt.sum(axis=(0, 1))))          # slice with the most tumour
        ax.imshow(volume[:, :, z].T, cmap="gray")
        ax.contour(gt[:, :, z].T, colors="red", linewidths=2)
        ax.contour(pred[:, :, z].T, colors="lime", linewidths=2, linestyles="dashed")
        ax.add_patch(plt.Rectangle(
            (bbox.x_min, bbox.y_min), bbox.x_max - bbox.x_min, bbox.y_max - bbox.y_min,
            edgecolor="cyan", facecolor="none", linewidth=1, linestyle=":",
        ))
        ax.set_title(f"fold {k}  {case['case']}  z={z}\ndice={gte.dice(pred, gt):.3f}", fontsize=8)
    gte._free(predictor)

fig.suptitle(f"{FINETUNED_LABEL} — red: GT,  green: prediction,  cyan: box prompt", y=1.0)
plt.tight_layout()
plt.show()